# Phase 3A: Read-Only Preprocessing Diagnostic

This notebook executes the **Phase 3A Read-Only Preprocessing Diagnostic** (`imagined_speech.data.preprocessing_diagnostic`) across all 690 EDF recordings discovered in Phase 1.

### Scientific Safeguards
- **Strictly Read-Only**: Raw `.edf` files, Phase 1 audit outputs (`phase1_audit/`), and Phase 2 signal-quality outputs (`phase2_signal_quality/`) are never modified. All Phase 3A outputs are written to a separate directory (`/content/drive/MyDrive/imagined_speech_outputs/phase3a_diagnostic/` in Colab or `./outputs/phase3a_diagnostic/` locally).
- **Candidate Filter Evaluation Only**: Per-channel mean subtraction followed by a 4th-order Butterworth `0.5-45.0 Hz` zero-phase band-pass filter at `128 Hz` (`sosfiltfilt`) is evaluated on derived in-memory copies only.
- **Exploratory Screening Terminology**: Candidate thresholds (`min_std_uV` in `{0.001, 0.25, 0.5, 1.0}`, `max_ptp_uV` in `{500, 1000, 1500, 2000}`, and recording policies allowing `0, 1, or 2` flagged channels) are evaluated without permanently excluding any channels or recordings. Recordings/channels satisfying a rule are reported strictly as **unflagged under that candidate rule**, not as verified clean data.
- **No Unverified Participant Assumptions**: Filename prefixes (`unverified_candidate_prefix`) are cross-tabulated against EDF header metadata without assuming they identify human participants.
- **Full-Recording Provenance**: No undocumented imagined-speech onset/offset sub-intervals are inferred or cropped.

## 1. Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository

In [ ]:
import subprocess

REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f'Cloning {REPO_URL} into {COLAB_REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO_DIR)], check=True)
    elif (COLAB_REPO_DIR / '.git').is_dir():
        print(f'Updating existing repository at {COLAB_REPO_DIR}...')
        subprocess.run(['git', '-C', str(COLAB_REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        raise RuntimeError(
            f'Directory {COLAB_REPO_DIR} exists but is not a valid Git repository.'
        )
    os.chdir(COLAB_REPO_DIR)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'pyproject.toml not found in working directory {REPO_ROOT}. '
        'Ensure the notebook is running from the repository root.'
    )

print(f'Repository root: {REPO_ROOT}')

## 3. Install the Package

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

src_dir = str((REPO_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import imagined_speech
print(f'Installed imagined_speech version: {imagined_speech.__version__}')

## 4. Verify Existing Phase 1 Audit Reports and Snapshot Phase 1 / Phase 2 Checksums
Confirm that `manifest.csv`, `channel_audit.csv`, and `audit_summary.json` exist and are non-empty, and snapshot existing Phase 1 / Phase 2 artifact checksums to verify at the end of the notebook that neither directory was modified.

In [ ]:
import hashlib
import json
import pandas as pd
from IPython.display import display
from imagined_speech.data.signal_quality import load_and_validate_manifest

PHASE1_AUDIT_DIR = Path(
    os.environ.get(
        'ISR_PHASE1_AUDIT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase1_audit'
        if IN_COLAB
        else './outputs/phase1_audit',
    )
)
PHASE2_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE2_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase2_signal_quality'
        if IN_COLAB
        else './outputs/phase2_signal_quality',
    )
)
PHASE3A_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3a_diagnostic'
        if IN_COLAB
        else './outputs/phase3a_diagnostic',
    )
)
CONFIG_PATH = REPO_ROOT / 'configs' / 'default_config.yaml'


def snapshot_dir_hashes(directory: Path) -> dict[str, str]:
    if not directory.is_dir():
        return {}
    hashes: dict[str, str] = {}
    for fpath in sorted(p for p in directory.rglob('*') if p.is_file()):
        rel = fpath.relative_to(directory).as_posix()
        hashes[rel] = hashlib.sha256(fpath.read_bytes()).hexdigest()
    return hashes


REQUIRED_PHASE1_FILES = ('manifest.csv', 'channel_audit.csv', 'audit_summary.json')
missing_phase1 = [
    name
    for name in REQUIRED_PHASE1_FILES
    if not (PHASE1_AUDIT_DIR / name).is_file() or (PHASE1_AUDIT_DIR / name).stat().st_size == 0
]
if missing_phase1:
    raise FileNotFoundError(
        f'Required Phase 1 audit reports are missing or empty in {PHASE1_AUDIT_DIR}: '
        f'{missing_phase1}. Please run notebooks/01_phase1_dataset_audit.ipynb first.'
    )

phase1_hashes_before = snapshot_dir_hashes(PHASE1_AUDIT_DIR)
phase2_hashes_before = snapshot_dir_hashes(PHASE2_OUTPUT_DIR)

manifest_df = load_and_validate_manifest(PHASE1_AUDIT_DIR / 'manifest.csv')
channel_audit_df = pd.read_csv(PHASE1_AUDIT_DIR / 'channel_audit.csv')
phase1_summary = json.loads((PHASE1_AUDIT_DIR / 'audit_summary.json').read_text(encoding='utf-8'))

print(f'Verified Phase 1 audit directory: {PHASE1_AUDIT_DIR} ({len(phase1_hashes_before)} files snapshotted)')
print(f'Phase 2 output directory files snapshotted: {len(phase2_hashes_before)}')
print(f'Phase 3A target output directory:           {PHASE3A_OUTPUT_DIR}')
print(f'Loaded manifest_df ({len(manifest_df)} rows) and channel_audit_df ({len(channel_audit_df)} rows).')

## 5. Verify or Extract Dataset Archive and Resolve `DATA_ROOT`
Use `/content/data/raw/Imagined_speech_EEG_edf` when already present in the runtime, or extract `/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip` into `/content/data/raw` in a fresh Colab runtime.

In [ ]:
import zipfile
from imagined_speech.exceptions import DatasetDiscoveryError

DRIVE_ZIP_PATH = Path(
    os.environ.get(
        'ISR_DRIVE_ZIP',
        '/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip',
    )
)
EXTRACT_ROOT = Path(
    os.environ.get(
        'ISR_EXTRACT_ROOT',
        '/content/data/raw' if IN_COLAB else './data/raw',
    )
)
CONFIRMED_COLAB_DATA_ROOT = Path('/content/data/raw/Imagined_speech_EEG_edf')
EXPECTED_TASKS = ('Char', 'Digit', 'Image')


def _has_expected_task_dirs(candidate: Path, expected_tasks: tuple[str, ...] = EXPECTED_TASKS) -> tuple[bool, list[str]]:
    if not candidate.is_dir():
        return False, []
    child_dirs = {
        child.name.lower(): child.name
        for child in sorted(candidate.iterdir(), key=lambda p: p.name.lower())
        if child.is_dir()
    }
    expected_lower = {t.lower() for t in expected_tasks}
    if expected_lower.issubset(child_dirs.keys()):
        return True, [child_dirs[t.lower()] for t in expected_tasks]
    return False, []


def locate_dataset_root(
    search_root: Path,
    expected_tasks: tuple[str, ...] = EXPECTED_TASKS,
) -> tuple[Path, list[str]]:
    if not search_root.exists() or not search_root.is_dir():
        raise DatasetDiscoveryError(
            f'Extraction directory does not exist or is not a directory: {search_root}'
        )
    candidate_dirs = [search_root] + sorted(
        (
            d
            for d in search_root.rglob('*')
            if d.is_dir() and not any(part.startswith('.') or part == '__MACOSX' for part in d.parts)
        ),
        key=lambda p: (len(p.parts), p.as_posix().lower()),
    )
    for cand in candidate_dirs:
        matched, folders = _has_expected_task_dirs(cand, expected_tasks)
        if matched:
            return cand.resolve(), folders
    raise DatasetDiscoveryError(
        f'Could not find a directory under {search_root} containing required '
        f'category folders {list(expected_tasks)}.'
    )


existing_edfs = (
    [p for p in EXTRACT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() == '.edf']
    if EXTRACT_ROOT.exists()
    else []
)
if existing_edfs:
    print(f'Skipping ZIP extraction: found {len(existing_edfs)} existing .edf files under {EXTRACT_ROOT}.')
else:
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(f'Dataset ZIP archive not found at {DRIVE_ZIP_PATH}.')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {DRIVE_ZIP_PATH} -> {EXTRACT_ROOT} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)

env_data_root = os.environ.get('ISR_DATA_ROOT')
if env_data_root:
    DATA_ROOT, discovered_categories = locate_dataset_root(Path(env_data_root))
elif IN_COLAB and _has_expected_task_dirs(CONFIRMED_COLAB_DATA_ROOT)[0]:
    DATA_ROOT = CONFIRMED_COLAB_DATA_ROOT.resolve()
    discovered_categories = _has_expected_task_dirs(CONFIRMED_COLAB_DATA_ROOT)[1]
else:
    DATA_ROOT, discovered_categories = locate_dataset_root(EXTRACT_ROOT)

print(f'Resolved DATA_ROOT:          {DATA_ROOT}')
print(f'Discovered category folders: {discovered_categories}')

## 6. Run Unit Tests and Diagnostic Smoke Test (1 Recording per Task)
Before processing all 690 recordings, run the unit test suite (`tests/test_preprocessing_diagnostic.py`) and execute an in-memory smoke test on 1 representative recording from each of `Char`, `Digit`, and `Image`.

In [ ]:
from imagined_speech.config import load_config
from imagined_speech.data.preprocessing_diagnostic import run_phase3a_diagnostic
from imagined_speech.data.signal_quality import select_representative_recordings
from imagined_speech.utils import set_global_seed, setup_logging

print('Running unit tests for Phase 3A diagnostic helpers...')
pytest_proc = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_preprocessing_diagnostic.py', '-v'],
    capture_output=True,
    text=True,
)
if pytest_proc.stdout:
    print(pytest_proc.stdout)
if pytest_proc.stderr:
    print(pytest_proc.stderr, file=sys.stderr)
if pytest_proc.returncode != 0:
    raise RuntimeError(
        f'Phase 3A unit tests failed with exit code {pytest_proc.returncode}. '
        'See captured pytest output above.'
    )

config = load_config(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    output_dir=PHASE3A_OUTPUT_DIR,
)
setup_logging(config.log_level)
set_global_seed(config.seed)

smoke_inputs = select_representative_recordings(
    manifest_df=manifest_df,
    data_root=config.data_root,
    expected_tasks=config.discovery.expected_tasks,
    recordings_per_task=1,
    specific_recordings=(),
    seed=config.seed,
)
smoke_paths = [s.relative_source_path for s in smoke_inputs]
print(f'Running Phase 3A smoke test on {len(smoke_paths)} representative recordings: {smoke_paths}')

smoke_result = run_phase3a_diagnostic(
    config=config,
    manifest_source=manifest_df,
    output_dir=PHASE3A_OUTPUT_DIR,
    selected_relative_paths=smoke_paths,
    save_outputs=False,
)
assert smoke_result.summary['n_recordings_inspected'] == 3
assert smoke_result.summary['n_channel_traces_inspected'] == 42
print('Smoke test passed! Pre- vs. Post-filter summary on 3 recordings (42 channel traces):')
display(
    smoke_result.channel_stats_df[
        [
            'task_category',
            'relative_source_path',
            'raw_channel_name',
            'pre_mean_uV',
            'pre_std_uV',
            'pre_ptp_uV',
            'post_mean_uV',
            'post_std_uV',
            'post_ptp_uV',
            'car_post_ptp_uV',
        ]
    ].head(14)
)

## 7. Run Full Phase 3A Read-Only Preprocessing Diagnostic Across All 690 Recordings
Evaluate the candidate `0.5-45 Hz` filter, threshold grid (`min_std_uV` in `{0.001, 0.25, 0.5, 1.0}`, `max_ptp_uV` in `{500, 1000, 1500, 2000}`, `max_flagged_channels_allowed` in `{0, 1, 2}`), filename-prefix & EDF metadata cross-tabulations, `none` vs. `CAR` comparisons, and windowing provenance across all 690 recordings, saving outputs to `PHASE3A_OUTPUT_DIR`.

In [ ]:
phase3a_result = run_phase3a_diagnostic(
    config=config,
    manifest_source=manifest_df,
    output_dir=PHASE3A_OUTPUT_DIR,
    save_outputs=True,
)
print('=== Phase 3A Diagnostic Summary (All 690 Recordings) ===')
print(json.dumps(phase3a_result.summary, indent=2))

## 8. Review Pre-Filter vs. Post-Filter Threshold Results & Per-Task / Per-Label Balance

In [ ]:
thr_df = phase3a_result.threshold_summary_df
lbl_df = phase3a_result.threshold_label_balance_df

print('=== Pre-Filter Exploratory Threshold Rules (Allow 0, 1, or 2 Flagged Channels) ===')
display(
    thr_df[thr_df['signal_stage'] == 'pre_filter'][
        [
            'min_std_uV',
            'max_ptp_uV',
            'max_flagged_channels_allowed',
            'n_flagged_channel_rows',
            'n_flagged_low_std_channels',
            'n_flagged_high_ptp_channels',
            'n_unflagged_recordings',
            'unflagged_recording_pct',
            'unflagged_char_pct',
            'unflagged_digit_pct',
            'unflagged_image_pct',
            'min_unflagged_per_label',
            'max_to_min_label_ratio',
        ]
    ]
)

print('\n=== Post-Filter (0.5-45 Hz) Exploratory Threshold Rules (Allow 0, 1, or 2 Flagged Channels) ===')
display(
    thr_df[thr_df['signal_stage'] == 'post_filter'][
        [
            'min_std_uV',
            'max_ptp_uV',
            'max_flagged_channels_allowed',
            'n_flagged_channel_rows',
            'n_flagged_low_std_channels',
            'n_flagged_high_ptp_channels',
            'n_unflagged_recordings',
            'unflagged_recording_pct',
            'unflagged_char_pct',
            'unflagged_digit_pct',
            'unflagged_image_pct',
            'min_unflagged_per_label',
            'max_to_min_label_ratio',
        ]
    ]
)

sample_rule = 'post_filter__std_ge_0.5uV__ptp_le_1000uV__allow_0_flagged_ch'
print(f'\nLowest unflagged label counts under candidate rule: {sample_rule}')
display(
    lbl_df[lbl_df['rule_id'] == sample_rule]
    .sort_values(['unflagged_recording_pct', 'task_category', 'normalized_label'])
    .head(20)
)

## 9. Review Filename-Prefix & EDF Metadata Audit, `None` vs. `CAR` Comparison, and Windowing Provenance

In [ ]:
print('=== Unverified Filename-Prefix & EDF Header Metadata Cross-Tabulation ===')
display(phase3a_result.prefix_metadata_df.head(30))

print('\n=== No Re-Referencing (none) vs. Common Average Reference (CAR) Summary ===')
car_df = phase3a_result.car_comparison_df
display(
    car_df.sort_values('n_channels_post_ptp_gt_1000uV', ascending=False).head(15)
)

print('\n=== Full-Recording Windowing Provenance Simulation (No Inferred Onset/Offset) ===')
display(phase3a_result.windowing_simulation_df)

## 10. Display Diagnostic Report, Figures, and Verify Output & Phase 1/2 Integrity

In [ ]:
from IPython.display import Image, Markdown

REQUIRED_PHASE3A_FILES = (
    'pre_vs_post_filter_channel_stats.csv',
    'threshold_retention_summary.csv',
    'threshold_class_balance_by_label.csv',
    'prefix_and_header_metadata_audit.csv',
    'reference_car_vs_none_comparison.csv',
    'windowing_provenance_simulation.csv',
    'phase3a_diagnostic_summary.json',
    'phase3a_diagnostic_report.md',
)

missing_or_empty: list[str] = []
verification_rows: list[dict[str, object]] = []
for filename in REQUIRED_PHASE3A_FILES:
    fpath = PHASE3A_OUTPUT_DIR / filename
    exists = fpath.is_file()
    size_bytes = fpath.stat().st_size if exists else 0
    if not exists or size_bytes == 0:
        missing_or_empty.append(f'{fpath} (exists={exists}, size={size_bytes})')
    verification_rows.append(
        {
            'artifact': filename,
            'path': str(fpath),
            'exists': exists,
            'size_bytes': size_bytes,
        }
    )

figures_dir = PHASE3A_OUTPUT_DIR / 'figures'
png_files = sorted(p for p in figures_dir.glob('*.png') if p.is_file() and p.stat().st_size > 0) if figures_dir.is_dir() else []
if len(png_files) < 4:
    missing_or_empty.append(f'{figures_dir} (expected 4 PNG figures, found {len(png_files)})')

if missing_or_empty:
    raise RuntimeError('Phase 3A output verification failed: ' + ', '.join(missing_or_empty))

phase1_hashes_after = snapshot_dir_hashes(PHASE1_AUDIT_DIR)
phase2_hashes_after = snapshot_dir_hashes(PHASE2_OUTPUT_DIR)
if phase1_hashes_before != phase1_hashes_after:
    raise RuntimeError('Integrity violation: Phase 1 audit files were modified!')
if phase2_hashes_before != phase2_hashes_after:
    raise RuntimeError('Integrity violation: Phase 2 signal-quality files were modified!')

print(f'Verified {len(REQUIRED_PHASE3A_FILES)} Phase 3A files and {len(png_files)} PNG figures in {PHASE3A_OUTPUT_DIR}.')
print('Confirmed Phase 1 and Phase 2 output directories are byte-for-byte unchanged.')
display(pd.DataFrame(verification_rows))

report_path = PHASE3A_OUTPUT_DIR / 'phase3a_diagnostic_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))

for fig_file in png_files:
    print(f'Figure: {fig_file}')
    display(Image(filename=str(fig_file)))